In [ ]:
!pip install transformer-lens torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 6.4 MB/s eta 0:00:00
  Created wheel for transformers-stream-generator: filename=transformers_stream_generator-0.0.5-py3-none-any.whl size=12426 sha256=5ffee2996d3bdc39b58e32a222b30fcf332fed2a708870b9a9e83ccab8289fe3
  Stored in directory: /root/.cache/pip/wheels/a8/58/d2/014cb67c3cc6def738c1b1635dbf4e3dab6fb63aba7070dce0
Successfully built transformers-stream-generator


In [ ]:
import torch
import numpy as np
from transformer_lens import HookedTransformer
from transformer_lens.patching import generic_activation_patch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = HookedTransformer.from_pretrained(
    "gpt2-small",
    device=device
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [ ]:
GLOBAL_HEADS = [
    (5,5),
    (7,10),
    (5,1),
    (6,9),
    (7,2),
    (5,0)
]

In [ ]:
prompts = [


    "The capital of India is",
    "The largest planet in the solar system is",

    "Once upon a time there was",
    "In a distant galaxy humanity discovered",


    "The reason climate change is difficult to solve is",
    "Artificial intelligence may become dangerous if",


    "Alice went to Paris. Bob went to London. Alice went to",
    "The cat sat on the mat. The dog sat on the",

    "The true meaning of life is",
    "Materialistic Wealth is not sufficient"
]

In [ ]:
def token_lrr(tokens, window_size=20):

    seq = tokens.squeeze().tolist()

    if len(seq) < window_size:
        return 0

    repeated = 0
    total = 0

    for i in range(window_size, len(seq)):

        current_token = seq[i]

        previous_window = seq[
            i-window_size:i
        ]

        if current_token in previous_window:
            repeated += 1

        total += 1

    return (
        repeated / total
        if total > 0
        else 0
    )

In [ ]:
def get_layer(hook):



    return int(
        hook.name.split(".")[1]
    )


def ablation_hook(z, hook):

    z = z.clone()

    layer = get_layer(hook)

    for L,H in GLOBAL_HEADS:

        if layer == L:
            z[:,:,H,:] = 0

    return z


def patch_hook(clean_cache, prompt_len):

    def hook_fn(z, hook):

        z = z.clone()

        layer = get_layer(hook)

        clean_z = clean_cache[
            hook.name
        ]

        patch_len = min(
            prompt_len,
            z.shape[1],
            clean_z.shape[1]
        )

        for L,H in GLOBAL_HEADS:

            if layer == L:

                z[
                    :,
                    :patch_len,
                    H,
                    :
                ] = clean_z[
                    :,
                    :patch_len,
                    H,
                    :
                ]

        return z

    return hook_fn

In [ ]:
recoveries = []

GEN_TOKENS = 100

for prompt in prompts:

    print(
        "\n"
        + "="*60
    )

    print(
        "PROMPT:"
    )

    print(
        prompt
    )

    tokens = (
        model.to_tokens(
            prompt
        )
        .to(device)
    )

    prompt_len = (
        tokens.shape[1]
    )


    model.reset_hooks()

    _, clean_cache = (
        model.run_with_cache(
            tokens
        )
    )

    clean_generated = (
        model.generate(
            tokens,
            max_new_tokens=
            GEN_TOKENS,
            temperature=1.0,
            verbose=False
        )
    )

    clean_lrr = token_lrr(
        clean_generated
    )

    model.reset_hooks()

    used_layers = set(
        layer
        for layer,_
        in GLOBAL_HEADS
    )

    for layer in used_layers:

        model.blocks[
            layer
        ].attn.hook_z.add_hook(
            ablation_hook
        )

    abl_generated = (
        model.generate(
            tokens,
            max_new_tokens=
            GEN_TOKENS,
            temperature=1.0,
            verbose=False
        )
    )

    abl_lrr = token_lrr(
        abl_generated
    )


    model.reset_hooks()

    for layer in used_layers:

        model.blocks[
            layer
        ].attn.hook_z.add_hook(
            ablation_hook
        )

    for layer in used_layers:

        model.blocks[
            layer
        ].attn.hook_z.add_hook(

            patch_hook(
                clean_cache,
                prompt_len
            )

        )

    patched_generated = (
        model.generate(
            tokens,
            max_new_tokens=
            GEN_TOKENS,
            temperature=1.0,
            verbose=False
        )
    )

    patched_lrr = token_lrr(
        patched_generated
    )


    denominator = (
        clean_lrr
        -
        abl_lrr
    )

    if abs(
        denominator
    ) < 1e-8:

        recovery = np.nan

    else:

        recovery = (

            patched_lrr
            -
            abl_lrr

        ) / denominator

    recoveries.append(
        recovery
    )

    print(
        "\nClean LRR:",
        round(
            clean_lrr,
            4
        )
    )

    print(
        "Ablated LRR:",
        round(
            abl_lrr,
            4
        )
    )

    print(
        "Patched LRR:",
        round(
            patched_lrr,
            4
        )
    )

    print(
        "Recovery:",
        round(
            recovery,
            4
        )
    )

model.reset_hooks()


PROMPT:
The capital of India is

Clean LRR: 0.2093
Ablated LRR: 0.6744
Patched LRR: 0.1512
Recovery: 1.125

PROMPT:
The largest planet in the solar system is

Clean LRR: 0.1461
Ablated LRR: 0.809
Patched LRR: 0.1011
Recovery: 1.0678

PROMPT:
Once upon a time there was

Clean LRR: 0.0805
Ablated LRR: 0.3448
Patched LRR: 0.1379
Recovery: 0.7826

PROMPT:
In a distant galaxy humanity discovered

Clean LRR: 0.1739
Ablated LRR: 0.4023
Patched LRR: 0.1149
Recovery: 1.2582

PROMPT:
The reason climate change is difficult to solve is

Clean LRR: 0.1316
Ablated LRR: 0.7889
Patched LRR: 0.1667
Recovery: 0.9466

PROMPT:
Artificial intelligence may become dangerous if

Clean LRR: 0.0909
Ablated LRR: 0.3523
Patched LRR: 0.125
Recovery: 0.8696

PROMPT:
Alice went to Paris. Bob went to London. Alice went to

Clean LRR: 0.1489
Ablated LRR: 0.6489
Patched LRR: 0.266
Recovery: 0.766

PROMPT:
The cat sat on the mat. The dog sat on the

Clean LRR: 0.172
Ablated LRR: 0.5054
Patched LRR: 0.1935
Recovery: 0.9

In [ ]:
valid = [

    r
    for r in recoveries

    if not np.isnan(r)

]

print(
    "\n"
    + "="*60
)

print(
    "MEAN:",
    np.mean(valid)
)

print(
    "STD:",
    np.std(valid)
)

print(
    "MIN:",
    np.min(valid)
)

print(
    "MAX:",
    np.max(valid)
)


MEAN: 0.964885579496802
STD: 0.15012956872788907
MIN: 0.7659574468085106
MAX: 1.2582056892778992
